# SENSE: characterize stage on Kaggle (T4 x2)

Run the cells top to bottom. Each step has one job and a check that says what a good result looks like.

**Before you start:** Session options -> Accelerator = GPU T4 x2, Internet = on, then restart the session so both apply. Add-ons -> Secrets must hold `HF_TOKEN` (toggle on for this notebook). Accept the Llama-3 license on Hugging Face first.

## Step 1. Check the GPU
Good: a table listing two Tesla T4 cards.

In [ ]:
!nvidia-smi

## Step 2. Load the Hugging Face token
Good: `HF_TOKEN loaded: True`. A `ConnectionError` means Internet is off for this session.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))

## Step 3. Pick a disk and clone the repo
`/kaggle/working` is only about 20 GB and cannot hold the CUDA torch install plus the checkpoints. This cell picks the disk with the most free space, puts the repo, uv cache and Hugging Face cache all on it (one disk lets uv hardlink instead of copy), then clones.

Good: a list of disks, a `Picked:` line, commit `0041895`, and three paths. It stops with a message if no disk has about 45 GB free (a rough estimate, not a measurement).

In [ ]:
import os, shutil, subprocess

free = {d: shutil.disk_usage(d).free / 1e9
        for d in ["/kaggle/temp", "/kaggle/working", "/tmp", "/root"]
        if os.path.isdir(d) and os.access(d, os.W_OK)}
for d, g in free.items():
    print(f"{d:18s} {g:8.1f} GB free")
best = max(free, key=free.get)
print(f"\nPicked: {best}")
if free[best] < 45:
    raise SystemExit(f"Only {free[best]:.1f} GB free; need about 45 GB.")

base = os.path.join(best, "sense-env")
repo = os.path.join(base, "sense")
os.environ["REPO_DIR"] = repo
os.environ["UV_CACHE_DIR"] = os.path.join(base, "uv-cache")
os.environ["HF_HOME"] = os.path.join(base, "hf-cache")
os.environ["HF_CACHE_DIR"] = os.environ["HF_HOME"]

shutil.rmtree(repo, ignore_errors=True)
os.makedirs(base, exist_ok=True)
subprocess.run(["git", "clone", "-q", "https://github.com/udshah31/sense.git", repo], check=True)
print(subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
print("REPO_DIR =", repo)

## Step 4. Install the packages
This replaces `scripts/gpu_setup.sh`. That script fails at its last step on Kaggle (it tries to reinstall torch from a `cu121` index that has no matching wheel, and targets the system Python). The venv torch is already a CUDA build, so that step is not needed.

Good: seven `=== uv sync: ... ===` blocks and no `error:` lines. Takes several minutes.

In [ ]:
%%bash
set -e
install_uv() { command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh; }
install_uv
export PATH="$HOME/.local/bin:$PATH"
cd "$REPO_DIR"
for pkg in data services/neural services/orchestrator services/symbolic services/rag eval experiments; do
  echo "=== uv sync: $pkg ==="
  (cd "$pkg" && uv sync -q)
done
echo "all packages installed"

## Step 5. Verify the venv can see the GPU
Good: something like `2.7.0+cu126 True 2 Tesla T4`. If it says `False`, stop and fix before Step 6.

In [ ]:
!cd {os.environ["REPO_DIR"]}/experiments && uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"

## Step 6. Run `characterize`
Loads Llama-3 8B, Qwen3 8B and Qwen3 1.7B one at a time, 4-bit, 200 development examples each. The first run downloads large checkpoints, so it is slow. Keep this tab open. The log is also saved to `/tmp/characterize.log`.

Good: `preflight OK`, then one `[characterize] <model>` line per checkpoint, then `done. Results in ...`.

In [ ]:
!cd {os.environ["REPO_DIR"]} && ./experiments/run_gpu_experiments.sh characterize 2>&1 | tee /tmp/characterize.log

## Step 7. Read the headline numbers
`threshold_spread.<scale>.max_over_min_ratio` is the headline. Near 1.0 means checkpoints calibrate to about the same threshold. Compare `raw` and `normalized`, then the two axes in `summary_by_axis`.

In [ ]:
import json
p = os.path.join(os.environ["REPO_DIR"], "results", "entropy_distribution_characterization.json")
r = json.load(open(p))
print(json.dumps(r["threshold_spread"], indent=2))
print(json.dumps(r["summary_by_axis"], indent=2))

## Step 8. Save the results
`/tmp` is wiped when the session ends. Copy the results into `/kaggle/working` (small files), then download them from the file panel or use Save Version.

In [ ]:
import glob
dst = "/kaggle/working/sense-results"
os.makedirs(dst, exist_ok=True)
for f in glob.glob(os.path.join(os.environ["REPO_DIR"], "results", "*characterization*.json")) + \
         glob.glob(os.path.join(os.environ["REPO_DIR"], "results", "run_metadata_*.json")) + ["/tmp/characterize.log"]:
    shutil.copy(f, dst)
print(os.listdir(dst))